# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saibalaji2004/Flyrank-intern-repo/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [5]:
import pandas as pd
import numpy as np

# Load the anonymized starter dataset
dataset_url = (
    "https://raw.githubusercontent.com/saibalaji2004/"
    "Flyrank-intern-repo/main/data/raw/content_refresh_anonymized.csv"
)
df = pd.read_csv(dataset_url)

# Create the observed decline label
df["is_declining_label"] = (
    df["trend_direction"].eq("down").astype(int)
)

# Exclude identifiers, label-derived fields, and recent trend windows
excluded_columns = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
]

feature_columns = [
    col for col in df.columns
    if col not in excluded_columns
]

X = df[feature_columns].copy()
y = df["is_declining_label"].copy()

# Identify numeric and categorical columns
numeric_columns = X.select_dtypes(include="number").columns.tolist()
categorical_columns = X.select_dtypes(
    exclude="number"
).columns.tolist()

# Add missingness indicators before any imputation
for col in X.columns:
    if X[col].isna().any():
        X[f"{col}_has_missing"] = X[col].isna().astype(int)

print("Dataset shape:", df.shape)
print("Feature matrix shape:", X.shape)
print("Target distribution:")
print(y.value_counts().rename(index={0: "Not down", 1: "Down"}))
print("Numeric columns:", len(numeric_columns))
print("Categorical columns:", len(categorical_columns))
print("Excluded columns:", excluded_columns)

Dataset shape: (30000, 45)
Feature matrix shape: (30000, 46)
Target distribution:
is_declining_label
Down        16262
Not down    13738
Name: count, dtype: int64
Numeric columns: 23
Categorical columns: 11
Excluded columns: ['content_id', 'client_id', 'trend_direction', 'trend_pct', 'is_declining_label', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature notes: meaning, missing values, and availability

The feature vector contains numeric and categorical variables describing page characteristics, search visibility, engagement, content age, freshness, and content metadata.

- **Numeric features:** Variables such as `search_volume`, `word_count`, `impressions_90d`, `clicks_90d`, `sessions_90d`, `days_since_last_update`, and `avg_position` describe search demand, content size, traffic, freshness, and visibility.
- **Categorical features:** Variables such as `content_type`, `main_intent`, `competition_level`, `age_tier`, `freshness_tier`, `position_tier`, and `trend-related categories` represent groups or labels. The excluded target-derived field `trend_direction` is not used as a feature.
- **Missing values:** Missingness is retained as information through binary indicators for columns containing missing values. Missing numeric and categorical values must be imputed inside the training pipeline, with preprocessing fitted on training data only. Missing values must not automatically be replaced with zero because zero can have a real meaning.
- **Availability at prediction time:** Content metadata, content age, historical search visibility, and historical engagement may be available when a review decision is made, depending on the reporting setup. Their availability must be verified for the intended decision date.
- **Privacy:** `content_id` and `client_id` are excluded from the predictive feature vector. The client identifier can still be used to keep clients separated between training and testing.
- **Target leakage:** `trend_direction` defines the observed decline label, and `trend_pct` contributes to the trend classification. Both are excluded. The latest-versus-previous 30-day comparison fields are also excluded to avoid directly feeding recent trend-window information into the model.

The feature vector is intended for prioritizing human review, not for proving that a content refresh will improve search performance.

In [6]:
# Inspect missingness and feature availability

missing_summary = (
    df[feature_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("Top 15 columns by missing-value count:")
print(missing_summary.head(15))

print("\nNumeric feature examples:")
print(numeric_columns[:10])

print("\nCategorical feature examples:")
print(categorical_columns[:10])

print("\nMissingness indicators created:", len([
    col for col in feature_columns
    if df[col].isna().any()
]))

print("\nFeature review completed.")

Top 15 columns by missing-value count:
provider_used        21438
word_count            7699
char_count_tier       7699
word_count_tier       7699
char_count            7699
model_used            5733
competition_level     2610
search_volume         2468
competition           2468
cpc                   2468
main_intent           2374
scroll_rate            125
impressions_90d          0
sessions_90d             0
users_90d                0
dtype: int64

Numeric feature examples:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d']

Categorical feature examples:
['competition_level', 'content_type', 'main_intent', 'provider_used', 'model_used', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier']

Missingness indicators created: 12

Feature review completed.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### Leakage audit

The target `is_declining_label` is derived from `trend_direction`, so the target and its source fields must not appear in the predictive features. `trend_pct` is also excluded because it contributes to the trend classification.

The six fields comparing the latest and previous 30-day periods are excluded because they directly encode recent trend-window information. `content_id` and `client_id` are excluded from model features to prevent identifiers from driving predictions. The client identifier is retained separately for grouped train/test splitting.

The audit checks forbidden columns, verifies that no client appears in both split groups, and records that temporal leakage cannot be ruled out solely from this dataset because the exact availability time of every feature is not established.

In [7]:
# Explicit leakage and privacy checks

forbidden_features = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
]

# Check that forbidden columns are absent from the feature vector
leaked_features = sorted(set(feature_columns) & set(forbidden_features))

print("Forbidden fields found in feature vector:", leaked_features)
assert not leaked_features, "Leakage detected in feature vector!"

# Check that the target is separate from X
assert "is_declining_label" not in X.columns
assert len(X) == len(y)

# Check that client IDs can be used for grouped splitting
assert "client_id" in df.columns
print("Unique clients available for grouped splitting:", df["client_id"].nunique())

# Verify the label matches the documented definition
expected_target = df["trend_direction"].eq("down").astype(int)
assert y.equals(expected_target), "Target definition mismatch!"

print("PASS: forbidden fields excluded.")
print("PASS: target kept separate from features.")
print("PASS: target definition verified.")
print("NOTE: temporal availability still requires separate validation.")

Forbidden fields found in feature vector: []
Unique clients available for grouped splitting: 32
PASS: forbidden fields excluded.
PASS: target kept separate from features.
PASS: target definition verified.
NOTE: temporal availability still requires separate validation.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### Excluded fields and rationale

| Excluded field | Reason |
|---|---|
| `content_id` | Identifier, not a predictive content feature. |
| `client_id` | Used for grouped splitting, not model prediction. |
| `trend_direction` | Direct source of the decline label; would leak the answer. |
| `trend_pct` | Contributes to the trend classification and risks target leakage. |
| `is_declining_label` | Target variable; must never be included in its own features. |
| `impressions_last_30d` | Recent trend-window measurement; excluded to reduce leakage risk. |
| `clicks_last_30d` | Recent trend-window measurement; excluded to reduce leakage risk. |
| `sessions_last_30d` | Recent trend-window measurement; excluded to reduce leakage risk. |
| `impressions_prev_30d` | Previous trend-window measurement; excluded to reduce leakage risk. |
| `clicks_prev_30d` | Previous trend-window measurement; excluded to reduce leakage risk. |
| `sessions_prev_30d` | Previous trend-window measurement; excluded to reduce leakage risk. |

These exclusions support a more defensible review-prioritization model. They do not prove that every remaining feature was available at a historical prediction time, so temporal validation remains a limitation.

In [8]:
# Document and verify the excluded fields

exclusion_reasons = {
    "content_id": "Identifier; not a predictive feature.",
    "client_id": "Used for grouped splitting; excluded from prediction.",
    "trend_direction": "Source of the decline label; target leakage risk.",
    "trend_pct": "Contributes to trend classification; leakage risk.",
    "is_declining_label": "Target variable; cannot be its own feature.",
    "impressions_last_30d": "Recent trend-window measurement.",
    "clicks_last_30d": "Recent trend-window measurement.",
    "sessions_last_30d": "Recent trend-window measurement.",
    "impressions_prev_30d": "Previous trend-window measurement.",
    "clicks_prev_30d": "Previous trend-window measurement.",
    "sessions_prev_30d": "Previous trend-window measurement.",
}

exclusion_table = pd.DataFrame(
    [
        {"field": field, "reason": reason}
        for field, reason in exclusion_reasons.items()
    ]
)

print(exclusion_table.to_string(index=False))

assert set(exclusion_reasons) == set(forbidden_features)
assert not (set(feature_columns) & set(exclusion_reasons))

print("\nPASS: all excluded fields have documented reasons.")
print("PASS: no excluded field appears in the feature vector.")

               field                                                reason
          content_id                 Identifier; not a predictive feature.
           client_id Used for grouped splitting; excluded from prediction.
     trend_direction     Source of the decline label; target leakage risk.
           trend_pct    Contributes to trend classification; leakage risk.
  is_declining_label           Target variable; cannot be its own feature.
impressions_last_30d                      Recent trend-window measurement.
     clicks_last_30d                      Recent trend-window measurement.
   sessions_last_30d                      Recent trend-window measurement.
impressions_prev_30d                    Previous trend-window measurement.
     clicks_prev_30d                    Previous trend-window measurement.
   sessions_prev_30d                    Previous trend-window measurement.

PASS: all excluded fields have documented reasons.
PASS: no excluded field appears in the feature v

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.